# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets with their @id and name.
record_sets = []
print("Available record sets:")
for rs in metadata.record_sets:
    print(f"  @id: {rs['@id']}, name: {rs.get('name', '<unnamed>')}")
    record_sets.append(rs['@id'])

if not record_sets:
    print("No record sets found in metadata. Attempting to discover from schema.")
    # Try to read schema manually
    # (If you see this message, check the JSON-LD and the mlcroissant version)

# Let's explore the fields and columns for each record set
fields_by_recordset = {}
columns_by_recordset = {}
for rs_obj in metadata.record_sets:
    rs_id = rs_obj['@id']
    fields = rs_obj.get('fields', [])
    print(f"\nRecord Set {rs_id} fields:")
    for f in fields:
        if isinstance(f, dict):
            print(f"  Field @id: {f['@id']}, name: {f.get('name', '<unnamed>')}")
        else:
            print(f"  Field @id: {f}")
    fields_by_recordset[rs_id] = fields
    if 'columns' in rs_obj:
        print("Columns:")
        for c in rs_obj['columns']:
            print(f"  Column @id: {c['@id']}, name: {c.get('name', '<unnamed>')}")
        columns_by_recordset[rs_id] = rs_obj['columns']

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For the FAIR² dataset, after overview, select main table record set by its @id (example):
# You should inspect the output above for all available record set @id's.
if record_sets:
    main_recordset_id = record_sets[0]
else:
    # Fallback: use known record set @id (manually set based on your exploration)
    main_recordset_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # Example from 'distribution' @id

# Extract records
dataframes = {}
for rs_id in record_sets:
    print(f"Loading data for {rs_id}...")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

if main_recordset_id in dataframes:
    df = dataframes[main_recordset_id]
    print(f"Loaded columns: {df.columns.tolist()}")
    display(df.head())
else:
    print(f"Record set {main_recordset_id} not loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# You should check the DataFrame columns and select a numeric field by its name (@id).
# Example numeric fields to try (replace with actual field from your overview if needed): abundance, MW (molecular weight), coverage
numeric_field = None
candidates = ['abundance', 'MW', 'coverage', 'PeptideCount']
if main_recordset_id in dataframes:
    df = dataframes[main_recordset_id]
    for c in candidates:
        if c in df.columns:
            numeric_field = c
            break

    if numeric_field is None:
        print("No candidate numeric field found. Columns detected:", df.columns.tolist())
    else:
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        print(f"Using numeric field: {numeric_field} with threshold: {threshold}\n")
        # Filtering
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by another available field, e.g. accession or description
        group_field = None
        candidate_groups = ['accession', 'description', 'Sample']
        for g in candidate_groups:
            if g in filtered_df.columns:
                group_field = g
                break

        if group_field:
            grouped_df = (
                filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            )
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No group field found in columns. Columns detected:", df.columns.tolist())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if main_recordset_id in dataframes and numeric_field:
    df = dataframes[main_recordset_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot with another numeric if available
    # Try MW vs abundance
    if 'MW' in df.columns and numeric_field != 'MW':
        plt.figure(figsize=(7, 5))
        sns.scatterplot(x=df['MW'], y=df[numeric_field])
        plt.xlabel('MW')
        plt.ylabel(numeric_field)
        plt.title('MW vs ' + numeric_field)
        plt.show()
elif main_recordset_id in dataframes:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.